# Adaptive Business Management Tutor
### Multi-Agent System with LangChain ReAct

**Architecture:** A supervisor agent receives the user's request, decides which sub-agent to invoke, and each sub-agent has its own set of tools it can call autonomously.

```
User Input
    │
    ▼
Supervisor Agent  ──── decides ────►  Quick Study Agent    (tools: retrieve, summarise, gen_quiz, export_pdf)
                                  └►  Full Scale Agent     (orchestrates 4 sub-agents below)
                                            │
                                            ├─ Knowledge Assessment Agent  (tools: retrieve, gen_quiz, run_quiz, save_result)
                                            ├─ Learning Path Agent         (tools: fetch_results, analyse_gaps, save_path)
                                            ├─ Adaptive Quiz Agent         (tools: gen_quiz, run_quiz, score, check_pass)
                                            └─ Mastery Tracking Agent      (tools: update_mastery, gen_summary, export_pdf)
```

**Stack:** LangChain · Groq (llama3-8b-8192) · ChromaDB · SQLite · ReportLab

---
Run cells top to bottom. Cell 6 is the interactive entry point.

## Cell 1 — Install dependencies

In [1]:
!pip freeze > requirements.txt

In [35]:
import subprocess, sys

packages = [
    "python-dotenv",
    "langchain",
    "langchain-groq",
    "langchain-community",
    "langchain-core",
    "sentence-transformers",
    "chromadb",
    "groq",
    "reportlab",
    "pypdf"
]
for p in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

print("All packages installed.")

All packages installed.


## Cell 2 — Imports & configuration

In [36]:
!pip install langchain langchain-groq langchain-community langchain-core -q

In [37]:
!pip install --upgrade chromadb

In [5]:
import os, json, sqlite3, uuid, textwrap
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# LangChain
from langchain_groq import ChatGroq
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

# Vector DB & embeddings
from sentence_transformers import SentenceTransformer
import chromadb

# PDF
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors

load_dotenv()

True

In [59]:


# ── Config ──────────────────────────────────────────────────────────────────
GROQ_MODEL      = "llama-3.3-70b-versatile"
EMBED_MODEL     = "all-MiniLM-L6-v2"
CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "knowledge_base"
DB_PATH         = "./tutor.db"
OUTPUT_DIR      = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
PASS_THRESHOLD  = 0.80
MAX_QUIZ_ROUNDS = 4



In [7]:
# ── Shared clients ───────────────────────────────────────────────────────────
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.2,
    max_tokens=2000,
)


In [ ]:
llm.invoke("Hello, world!")

In [60]:
embed_model   = SentenceTransformer(EMBED_MODEL)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection(name=COLLECTION_NAME)


In [61]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [67]:
# ── Session state (single user) ──────────────────────────────────────────────
# This dict is passed between agents so they share context
SESSION = {
    "profile":          None,
    "context":          "",
    "diagnostic_score": None,
    "gap_data":         None,
    "adaptive_result":  None,
    "summary": None
}

print(f"LLM ready: {GROQ_MODEL}")
print(f"ChromaDB chunks: {collection.count()}")

LLM ready: llama-3.3-70b-versatile
ChromaDB chunks: 95


## Cell 3 — Database schema

In [63]:
def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS learner_profiles (
            id         TEXT PRIMARY KEY,
            name       TEXT NOT NULL,
            topic      TEXT NOT NULL,
            level      TEXT DEFAULT 'beginner',
            created_at TEXT DEFAULT (datetime('now'))
        );
        CREATE TABLE IF NOT EXISTS quiz_sessions (
            id             TEXT PRIMARY KEY,
            learner_id     TEXT NOT NULL,
            session_type   TEXT NOT NULL,
            topic          TEXT NOT NULL,
            score_pct      REAL,
            passed         INTEGER DEFAULT 0,
            round_number   INTEGER DEFAULT 1,
            responses_json TEXT,
            created_at     TEXT DEFAULT (datetime('now'))
        );
        CREATE TABLE IF NOT EXISTS mastery_records (
            id         TEXT PRIMARY KEY,
            learner_id TEXT NOT NULL,
            topic      TEXT NOT NULL,
            mastered   INTEGER DEFAULT 0,
            best_score REAL DEFAULT 0.0,
            updated_at TEXT DEFAULT (datetime('now'))
        );
    """)
    conn.commit()
    conn.close()

init_db()
print("Database ready:", DB_PATH)

Database ready: ./tutor.db


## Cell 4 — Tool functions

Every tool is a plain Python function first, then wrapped with `langchain_core.tools.Tool`.
Agents call these by name — the LLM decides *which* tool to use and *what* to pass to it.

In [64]:
# ── 1. Profile tools ────────────────────────────────────────────────────────

def search_learner_by_name(name: str) -> str:
    """
    Input: learner name string.
    Searches for an existing profile by name. Returns profile JSON if found, else error message.
    """
    name = name.strip()
    conn = sqlite3.connect(DB_PATH)
    row  = conn.execute(
        "SELECT id,name,topic,level FROM learner_profiles WHERE name=? ORDER BY created_at DESC LIMIT 1",
        (name,)
    ).fetchone()
    conn.close()
    if row:
        profile = dict(zip(["id","name","topic","level"], row))
        SESSION["profile"] = profile
        return json.dumps({"found": True, "profile": profile})
    return json.dumps({"found": False, "message": f"No existing profile for: {name}"})


def create_learner_profile(input_str: str) -> str:
    """
    Input: "name|topic|level"  e.g. "Alice|Financial Statements|beginner"
    Creates a learner profile in SQLite. Returns the profile as JSON string.
    """
    parts = [p.strip() for p in input_str.split("|")]
    name  = parts[0] if len(parts) > 0 else "Learner"
    topic = parts[1] if len(parts) > 1 else "Business Management"
    level = parts[2] if len(parts) > 2 else "beginner"
    if level not in ("beginner", "intermediate", "advanced"):
        level = "beginner"

    learner_id = str(uuid.uuid4())[:8]
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        "INSERT INTO learner_profiles (id,name,topic,level) VALUES (?,?,?,?)",
        (learner_id, name, topic, level)
    )
    conn.commit(); conn.close()

    profile = {"id": learner_id, "name": name, "topic": topic, "level": level}
    SESSION["profile"] = profile
    return json.dumps(profile)


def fetch_learner_profile(learner_id: str) -> str:
    """
    Input: learner_id string.
    Returns the learner profile as a JSON string, or an error message.
    """
    conn = sqlite3.connect(DB_PATH)
    row  = conn.execute(
        "SELECT id,name,topic,level FROM learner_profiles WHERE id=?",
        (learner_id.strip(),)
    ).fetchone()
    conn.close()
    if row:
        profile = dict(zip(["id","name","topic","level"], row))
        SESSION["profile"] = profile
        return json.dumps(profile)
    return f"No profile found for ID: {learner_id}"

In [65]:
def retrieve_study_material(topic: str) -> str:
    """
    Input: topic string (e.g. "Financial Statements").
    Retrieves the most relevant chunks from ChromaDB.
    Returns the joined context text.
    """
    topic = topic.strip()
    if collection.count() == 0:
        return f"No study material found in knowledge base for: {topic}"
    emb  = embed_model.encode([topic])[0].tolist()
    res  = collection.query(query_embeddings=[emb], n_results=min(15, collection.count()))
    ctx  = "\n\n".join(res["documents"][0])
    SESSION["context"] = ctx
    print(f"Retrieved {len(res['documents'][0])} chunks for topic '{topic}'. Total context size: {len(ctx)} chars.")
    return ctx

In [15]:
returns = retrieve_study_material("Financial Statements")
returns[:500]

Retrieved 8 chunks for topic 'Financial Statements'.


' the\nmanagement to these parties interested in the organisation and help in taking\nappropriate economic decisions. It may be noted that the financial statements\nconstitute an integral part of the annual report of the company in addition to\nthe directors report, auditors report, corporate governance report, and\nmanagement discussion and analysis.\nThe various uses and importance of financial statements are as follows:\n1. Report on stewardship function:  Financial statements report the\nperformance '

In [69]:
# ── 2. Retrieval tool ────────────────────────────────────────────────────────

def retrieve_study_material(topic: str) -> str:
    """
    Input: topic string (e.g. "Financial Statements").
    Retrieves the most relevant chunks from ChromaDB.
    Returns the joined context text.
    """
    topic = topic.strip()
    if collection.count() == 0:
        return f"No study material found in knowledge base for: {topic}"
    emb  = embed_model.encode([topic])[0].tolist()
    res  = collection.query(query_embeddings=[emb], n_results=min(15, collection.count()))
    ctx  = "\n\n".join(res["documents"][0])
    SESSION["context"] = ctx
    print(f"Retrieved {len(res['documents'][0])} chunks for topic '{topic}'.")
    print(f"Context preview:\n{ctx[:500]}...")
    return ctx

# ── 9. Generate summary tool ──────────────────────────────────────────────────

def generate_study_summary(topic: str) -> str:
    """
    Input: topic string.
    Generates main points, overview summary, and key exam questions.
    Returns JSON string with keys: main_points, summary, key_questions.
    """
    topic = topic.strip()
    ctx   = SESSION.get("context") or retrieve_study_material(topic)

    profile = SESSION.get("profile", {"level":"beginner"})
    prompt  = f"""You are a business management tutor making a detailed study aid for a {profile['level']} student.
Topic: {topic}

Using ONLY the context below, produce:
1. main_points: 15-30 key takeaways as detailed bullet strings, be concise but comprehensive, covering all major concepts. Each point should be 1-3 sentences.
2. summary: comprehensive 2-6 paragraph overview covering the topic depth, based on the complexity of the material and learner level.
3. key_questions: 5-8 important exam-style questions the student should be able to answer, with hints if possible.

Be thorough and comprehensive. Cover all major concepts from the material.

Context:
{ctx[:8000]}

Return ONLY valid JSON — no markdown:
{{"main_points":["..."],"summary":"...","key_questions":["..."]}}"""

    from groq import Groq
    raw_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    resp = raw_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4,
        max_tokens=4000,
    )
    raw = (resp.choices[0].message.content or "").strip()
    s   = raw[raw.find("{"):raw.rfind("}")+1]
    try:
        print(s)
        SESSION["summary"] = json.loads(s)
        return s
        
    except Exception:
        return json.dumps({"main_points":[],"summary":f"Study notes for {topic}.","key_questions":[]})


# ── 10. Export PDF tool ───────────────────────────────────────────────────────

def export_report_pdf(report_type: str) -> str:
    """
    Input: "quick_study" or "mastery_report"
    Reads SESSION for profile, summary, quiz results, gaps.
    Builds and saves a PDF. Returns the file path.
    """
    report_type = report_type.strip().lower()
    profile     = SESSION.get("profile", {"name":"Learner","topic":"Unknown","level":"beginner","id":"0"})

    # Get summary data
    summary_raw  = generate_study_summary(profile["topic"])
    try:
        summary_data = json.loads(summary_raw)
    except Exception:
        summary_data = {"main_points":[],"summary":"","key_questions":[]}

    title    = "Quick Study Guide" if report_type == "quick_study" else "Mastery Report"
    filename = f"{report_type}_{profile['name'].replace(' ','_')}_{profile['topic'].replace(' ','_')[:20]}.pdf"
    filepath = OUTPUT_DIR / filename

    # ── Build PDF ────────────────────────────────────────────────────────────
    doc    = SimpleDocTemplate(str(filepath), pagesize=letter,
                               leftMargin=inch, rightMargin=inch,
                               topMargin=inch, bottomMargin=inch)
    styles = getSampleStyleSheet()

    title_s   = ParagraphStyle("T2",   parent=styles["Title"],   fontSize=20, spaceAfter=6)
    h1_s      = ParagraphStyle("H1",   parent=styles["Heading1"],fontSize=14, spaceAfter=4, spaceBefore=14)
    body_s    = ParagraphStyle("B2",   parent=styles["Normal"],  fontSize=11, spaceAfter=4, leading=16)
    bullet_s  = ParagraphStyle("Bul",  parent=styles["Normal"],  fontSize=11, spaceAfter=3, leftIndent=18, leading=15)
    meta_s    = ParagraphStyle("Meta", parent=styles["Normal"],  fontSize=9,  textColor=colors.grey, spaceAfter=2)
    ok_s      = ParagraphStyle("OK",   parent=styles["Normal"],  fontSize=10, textColor=colors.HexColor("#1a7a1a"), spaceAfter=2)
    bad_s     = ParagraphStyle("Bad",  parent=styles["Normal"],  fontSize=10, textColor=colors.HexColor("#cc0000"), spaceAfter=2)
    exp_s     = ParagraphStyle("Exp",  parent=styles["Normal"],  fontSize=10, textColor=colors.HexColor("#555555"), leftIndent=18, spaceAfter=6)

    def HR():
        return HRFlowable(width="100%", thickness=0.5, color=colors.HexColor("#cccccc"), spaceAfter=6)

    story = []
    story.append(Paragraph(f"{title}: {profile['topic']}", title_s))
    story.append(Paragraph(
        f"Learner: {profile['name']}  |  Level: {profile['level']}  |  {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        meta_s))
    story.append(HR())

    story.append(Paragraph("Overview", h1_s))
    story.append(Paragraph(summary_data.get("summary",""), body_s))
    story.append(Spacer(1,6))

    story.append(Paragraph("Key Points", h1_s))
    for pt in summary_data.get("main_points",[]):
        story.append(Paragraph(f"• {pt}", bullet_s))
    story.append(Spacer(1,6))

    story.append(Paragraph("Important Questions to Review", h1_s))
    for i,kq in enumerate(summary_data.get("key_questions",[]),1):
        story.append(Paragraph(f"{i}. {kq}", bullet_s))

    # Mastery report extras
    if report_type == "mastery_report":
        score_data = (SESSION.get("diagnostic_score") or {})
        if score_data:
            story.append(HR())
            story.append(Paragraph("Assessment Results", h1_s))
            pct    = score_data.get("score_pct",0)*100
            passed = score_data.get("passed",False)
            story.append(Paragraph(
                f"Score: {score_data.get('correct',0)}/{score_data.get('total',0)} ({pct:.0f}%) — {'PASSED' if passed else 'NOT YET PASSED'}",
                body_s))

        gap_data = SESSION.get("gap_data")
        if gap_data and gap_data.get("gaps"):
            story.append(HR())
            story.append(Paragraph("Knowledge Gaps", h1_s))
            story.append(Paragraph(gap_data.get("summary",""), body_s))
            for g in gap_data.get("gaps",[]):
                story.append(Paragraph(f"• {g}", bullet_s))

            lp = gap_data.get("learning_path",[])
            if lp:
                story.append(HR())
                story.append(Paragraph("Your Personalised Learning Path", h1_s))
                for step in lp:
                    story.append(Paragraph(
                        f"{step['order']}. <b>{step['subtopic']}</b> — {step['why']}",
                        bullet_s))

    doc.build(story)
    print(f"  PDF exported: {filepath}")
    return str(filepath)


# ── 3. Quiz generation tool ──────────────────────────────────────────────────

def generate_quiz_questions(input_str: str) -> str:
    """
    Input: "topic|level|n_questions|missed_concepts"
      - missed_concepts is optional comma-separated list to focus on.
    Generates MCQs and returns them as a JSON string.
    """
    parts           = [p.strip() for p in input_str.split("|")]
    topic           = parts[0] if len(parts) > 0 else SESSION.get("profile", {}).get("topic", "Business Management")
    level           = parts[1] if len(parts) > 1 else "beginner"
    n               = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 5
    missed_raw      = parts[3] if len(parts) > 3 else ""
    missed          = [m.strip() for m in missed_raw.split(",")] if missed_raw else []

    ctx = SESSION.get("context") or retrieve_study_material(topic)

    difficulty_map = {
        "beginner":     "60% easy, 40% medium",
        "intermediate": "20% easy, 60% medium, 20% hard",
        "advanced":     "20% medium, 80% hard",
    }
    diff = difficulty_map.get(level, "40% easy, 60% medium")

    focus = f"\nFocus especially on these concepts the learner struggled with: {missed}\n" if missed else ""

    prompt = f"""You are a business management examiner.
Generate exactly {n} multiple choice questions about: {topic}
Learner level: {level} — difficulty mix: {diff}
{focus}
Mix question types: conceptual, scenario-based, definition, compare/contrast, true-or-false-as-MCQ.
Each question must have 4 options (A B C D) and exactly one correct answer.
Make questions detailed and comprehensive, testing deep understanding.
Base every question strictly on the context. No invented facts.

Context:
{ctx[:8000]}

Return ONLY a valid JSON array. No markdown, no preamble.
Each item: {{"id":1,"type":"conceptual","difficulty":"easy","question":"...","options":{{"A":"...","B":"...","C":"...","D":"..."}},"correct_answer":"A","explanation":"one sentence why"}}"""

    from groq import Groq
    raw_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    resp = raw_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5,
        max_tokens=2000,
    )
    raw = (resp.choices[0].message.content or "").strip()

    # Clean fences
    if raw.startswith("```"):
        lines = raw.splitlines()
        raw = "\n".join(lines[1:])
        if raw.rstrip().endswith("```"):
            raw = raw.rstrip()[:-3]
    start = raw.find("["); end = raw.rfind("]")
    if start != -1 and end != -1:
        raw = raw[start:end+1]

    try:
        questions = json.loads(raw.strip())
        # Validate
        for q in questions:
            if q.get("correct_answer") not in q.get("options", {}):
                raise ValueError(f"correct_answer not in options: Q{q.get('id')}")
        return json.dumps(questions)
    except Exception as e:
        return json.dumps({"error": str(e), "raw": raw[:200]})




In [51]:
# ═══════════════════════════════════════════════════════════════════════════
# ── TOOL FUNCTIONS ─────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════






# ── 4. Interactive quiz runner ────────────────────────────────────────────────

def run_interactive_quiz(questions_json: str) -> str:
    """
    Input: JSON string of questions (from generate_quiz_questions).
    Presents questions interactively in the notebook.
    Returns responses as a JSON string.
    """
    try:
        questions = json.loads(questions_json)
        if isinstance(questions, dict) and "error" in questions:
            return json.dumps({"error": questions["error"]})
    except Exception as e:
        return json.dumps({"error": f"Could not parse questions: {e}"})

    responses = []
    for q in questions:
        print(f"\n  [{q.get('difficulty','?').upper()}] [{q.get('type','MCQ')}] Q{q['id']}:")
        print(f"  {q['question']}\n")
        for letter, text in q["options"].items():
            print(f"    {letter}) {text}")

        while True:
            ans = input("\n  Your answer (A/B/C/D): ").strip().upper()
            if ans in ("A","B","C","D"):
                break
            print("  Please enter A, B, C, or D.")

        correct = ans == q["correct_answer"]
        if correct:
            print("  ✓ Correct!")
        else:
            print("  ✗ Incorrect.")
            print(f"  Correct answer: {q['correct_answer']}) {q['options'][q['correct_answer']]}")
            print(f"  Why: {q.get('explanation','')}")

        responses.append({
            "question_id":    q["id"],
            "question":       q["question"],
            "difficulty":     q.get("difficulty","?"),
            "type":           q.get("type","MCQ"),
            "user_answer":    ans,
            "correct_answer": q["correct_answer"],
            "correct_option": q["options"][q["correct_answer"]],
            "explanation":    q.get("explanation",""),
            "is_correct":     correct,
        })

    return json.dumps(responses)


# ── 5. Score & save tool ─────────────────────────────────────────────────────

def score_and_save_quiz(input_str: str) -> str:
    """
    Input: "session_type|round_number|responses_json"
    Scores the responses, saves to DB, returns score summary JSON.
    """
    parts        = input_str.split("|", 2)
    session_type = parts[0].strip() if len(parts) > 0 else "quiz"
    round_num    = int(parts[1].strip()) if len(parts) > 1 and parts[1].strip().isdigit() else 1
    resp_json    = parts[2].strip() if len(parts) > 2 else "[]"

    try:
        responses = json.loads(resp_json)
    except Exception:
        return json.dumps({"error": "Could not parse responses JSON"})

    total   = len(responses)
    correct = sum(1 for r in responses if r.get("is_correct"))
    pct     = correct / total if total else 0
    passed  = pct >= PASS_THRESHOLD
    gaps    = [r for r in responses if not r.get("is_correct")]

    score = {
        "total":            total,
        "correct":          correct,
        "score_pct":        round(pct, 3),
        "passed":           passed,
        "missed_concepts":  [r["question"][:80] for r in gaps],
        "gaps":             gaps,
    }

    profile = SESSION.get("profile")
    if profile:
        session_id = str(uuid.uuid4())[:8]
        conn = sqlite3.connect(DB_PATH)
        conn.execute(
            """INSERT INTO quiz_sessions
               (id,learner_id,session_type,topic,score_pct,passed,round_number,responses_json)
               VALUES (?,?,?,?,?,?,?,?)""",
            (session_id, profile["id"], session_type, profile["topic"],
             pct, int(passed), round_num, json.dumps(responses))
        )
        conn.commit(); conn.close()

    SESSION["diagnostic_score"] = score
    print(f"\n  Score: {correct}/{total} ({pct*100:.0f}%) — {'PASSED ✓' if passed else 'not yet passed'}")
    return json.dumps(score)


# ── 6. Gap analysis tool ──────────────────────────────────────────────────────

def analyse_knowledge_gaps(responses_json: str) -> str:
    """
    Input: JSON string of quiz responses.
    Calls the LLM to identify knowledge gaps and produce an ordered learning path.
    Returns gap analysis as JSON string.
    """
    try:
        responses = json.loads(responses_json)
    except Exception:
        return json.dumps({"error": "Could not parse responses"})

    wrong = [r for r in responses if not r.get("is_correct")]
    if not wrong:
        result = {"gaps":[],"learning_path":[],"summary":"No gaps found — excellent work!"}
        SESSION["gap_data"] = result
        return json.dumps(result)

    profile = SESSION.get("profile", {"level":"beginner","topic":"Business Management"})
    wrong_qs = "\n".join(
        f"- Q: {r['question']} | Chose: {r['user_answer']} | Correct: {r['correct_answer']}) {r['correct_option']}"
        for r in wrong
    )

    prompt = f"""A {profile['level']} student studying '{profile['topic']}' answered these questions incorrectly:
{wrong_qs}

1. Identify the specific knowledge gaps (name the exact concepts not understood).
2. Create an ordered learning path of sub-topics to study, starting with foundational gaps.
3. Write one sentence explaining why each sub-topic matters.

Return ONLY valid JSON — no markdown:
{{"gaps":["concept 1","concept 2"],"learning_path":[{{"order":1,"subtopic":"...","why":"..."}}],"summary":"one sentence overall assessment"}}"""

    from groq import Groq
    raw_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    resp = raw_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3,
        max_tokens=800,
    )
    raw = (resp.choices[0].message.content or "").strip()
    s   = raw[raw.find("{"):raw.rfind("}")+1]
    try:
        result = json.loads(s)
    except Exception:
        result = {
            "gaps":          [r["question"][:60] for r in wrong],
            "learning_path": [{"order":i+1,"subtopic":r["question"][:60],"why":"Answered incorrectly"}
                               for i,r in enumerate(wrong)],
            "summary":       f"{len(wrong)} concepts need review."
        }

    SESSION["gap_data"] = result

    # Save learning path JSON
    if SESSION.get("profile"):
        pf   = OUTPUT_DIR / f"learning_path_{SESSION['profile']['id']}.json"
        with open(pf,"w") as f:
            json.dump({"profile":profile,"gap_data":result,"generated_at":datetime.now().isoformat()},f,indent=2)
        print(f"  Learning path saved: {pf}")

    print(f"  Gap analysis: {result['summary']}")
    print(f"  Gaps identified: {result['gaps']}")
    return json.dumps(result)


# ── 7. Check pass / loop control ─────────────────────────────────────────────

def check_pass_status(score_json: str) -> str:
    """
    Input: score JSON string from score_and_save_quiz.
    Returns a plain-English status the agent uses to decide whether to loop.
    """
    try:
        score = json.loads(score_json) if isinstance(score_json, str) else score_json
    except Exception:
        return "Could not parse score."
    pct    = score.get("score_pct", 0) * 100
    passed = score.get("passed", False)
    missed = score.get("missed_concepts", [])
    if passed:
        return f"PASSED with {pct:.0f}%. Proceed to mastery tracking."
    return (f"NOT PASSED — score {pct:.0f}%. "
            f"Missed concepts: {missed[:3]}. Generate new questions focusing on these gaps.")


# ── 8. Update mastery tool ───────────────────────────────────────────────────

def update_mastery_record(input_str: str) -> str:
    """
    Input: "score_pct|passed"  e.g. "0.85|True"
    Updates mastery record in DB and upgrades learner level if passed.
    Returns confirmation string.
    """
    parts  = input_str.split("|")
    pct    = float(parts[0].strip()) if len(parts) > 0 else 0.0
    passed = parts[1].strip().lower() == "true" if len(parts) > 1 else False

    profile = SESSION.get("profile")
    if not profile:
        return "No active profile in session."

    conn = sqlite3.connect(DB_PATH)
    existing = conn.execute(
        "SELECT best_score FROM mastery_records WHERE learner_id=? AND topic=?",
        (profile["id"], profile["topic"])
    ).fetchone()
    best = max((existing[0] if existing else 0), pct)
    if existing:
        conn.execute(
            "UPDATE mastery_records SET mastered=?,best_score=?,updated_at=datetime('now') WHERE learner_id=? AND topic=?",
            (int(passed), best, profile["id"], profile["topic"])
        )
    else:
        conn.execute(
            "INSERT INTO mastery_records (id,learner_id,topic,mastered,best_score) VALUES (?,?,?,?,?)",
            (str(uuid.uuid4())[:8], profile["id"], profile["topic"], int(passed), best)
        )
    if passed:
        conn.execute(
            "UPDATE learner_profiles SET level='intermediate' WHERE id=?",
            (profile["id"],)
        )
        SESSION["profile"]["level"] = "intermediate"
    conn.commit(); conn.close()

    status = "MASTERED" if passed else "IN PROGRESS"
    return f"Mastery record updated. Status: {status}. Best score: {best*100:.0f}%."



print("All tool functions defined.")


All tool functions defined.


## Cell 5 — Wrap tools & build agents

Each agent gets:
- A set of `Tool` objects it can call
- A ReAct prompt (from LangChain hub) that tells it to reason → act → observe
- An `AgentExecutor` that runs the loop

In [18]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

# Pull the standard ReAct prompt once
react_prompt = hub.pull("hwchase17/react")


c:\Users\Kurinchi1\anaconda3\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [53]:

# ═══════════════════════════════════════════════════════════════════════════
# ── WRAP FUNCTIONS AS LANGCHAIN TOOLS ──────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════

# Wrap tools with StructuredTool for proper schema handling with Groq
tool_create_profile = StructuredTool.from_function(
    func=create_learner_profile,
    name="create_learner_profile",
    description=(
        "Create a new learner profile. "
        "Input must be a string in this format: 'name|topic|level'. "
        "Level must be beginner, intermediate, or advanced. "
        "Returns a JSON object with the profile including a learner ID."
    ),
)

tool_fetch_profile = StructuredTool.from_function(
    func=fetch_learner_profile,
    name="fetch_learner_profile",
    description="Fetch an existing learner profile by their learner ID string. Returns profile JSON.",
)

tool_retrieve = StructuredTool.from_function(
    func=retrieve_study_material,
    name="retrieve_study_material",
    description=(
        "Retrieve relevant study material from the knowledge base for a given topic. "
        "Input is the topic string. Returns context text used for quiz generation and summaries."
    ),
)

tool_generate_quiz = StructuredTool.from_function(
    func=generate_quiz_questions,
    name="generate_quiz_questions",
    description=(
        "Generate multiple choice quiz questions. "
        "Input format: 'topic|level|n_questions|missed_concepts' "
        "where missed_concepts is optional comma-separated concepts to focus on. "
        "Example: 'Financial Statements|beginner|5|balance sheet,accruals'. "
        "Returns a JSON array of question objects."
    ),
)

tool_run_quiz = StructuredTool.from_function(
    func=run_interactive_quiz,
    name="run_interactive_quiz",
    description=(
        "Present quiz questions to the learner interactively and collect their answers. "
        "Input is the JSON string of questions from generate_quiz_questions. "
        "Returns a JSON array of response objects including whether each answer was correct."
    ),
)

tool_score = StructuredTool.from_function(
    func=score_and_save_quiz,
    name="score_and_save_quiz",
    description=(
        "Score quiz responses and save the session to the database. "
        "Input format: 'session_type|round_number|responses_json' "
        "where session_type is 'diagnostic' or 'adaptive'. "
        "Example: 'adaptive|2|[...]'. "
        "Returns a score summary JSON with total, correct, score_pct, passed, and missed_concepts."
    ),
)

tool_analyse_gaps = StructuredTool.from_function(
    func=analyse_knowledge_gaps,
    name="analyse_knowledge_gaps",
    description=(
        "Analyse quiz responses to identify knowledge gaps and produce a personalised learning path. "
        "Input is the JSON string of quiz responses. "
        "Returns gap analysis JSON with gaps list, ordered learning_path, and summary."
    ),
)

tool_check_pass = StructuredTool.from_function(
    func=check_pass_status,
    name="check_pass_status",
    description=(
        "Check whether the learner passed the quiz (80% threshold). "
        "Input is the score JSON string from score_and_save_quiz. "
        "Returns a plain-English status string: PASSED or NOT PASSED with guidance on next step."
    ),
)

tool_update_mastery = StructuredTool.from_function(
    func=update_mastery_record,
    name="update_mastery_record",
    description=(
        "Update the learner's mastery record in the database after completing the adaptive quiz. "
        "Input format: 'score_pct|passed' e.g. '0.85|True'. "
        "Returns a confirmation string."
    ),
)

tool_summary = StructuredTool.from_function(
    func=generate_study_summary,
    name="generate_study_summary",
    description=(
        "Generate a study summary for a topic including main points, overview, and key exam questions. "
        "Input is the topic string. "
        "Returns JSON with main_points, summary, and key_questions."
    ),
)

tool_export_pdf = StructuredTool.from_function(
    func=export_report_pdf,
    name="export_report_pdf",
    description=(
        "Export a PDF report for the current learner session. "
        "Input must be either 'quick_study' or 'mastery_report'. "
        "Returns the file path of the saved PDF."
    ),
)

tool_search_profile = StructuredTool.from_function(
    func=search_learner_by_name,
    name="search_learner_by_name",
    description=(
        "Search for an existing learner profile by name. "
        "Input is the learner name string. "
        "Returns JSON with found status and profile details if found."
    ),
)

# ═══════════════════════════════════════════════════════════════════════════
# ── BUILD AGENTS ───────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════

def make_agent(tools: list, system_note: str = "") -> AgentExecutor:
    """Helper: build a ReAct AgentExecutor with the given tools."""
    # Create agent directly without bind_tools - StructuredTool handles schema correctly
    agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)
    return AgentExecutor.from_agent_and_tools(
        agent=agent,
        tools=tools,
        verbose=False,
        max_iterations=20,
        handle_parsing_errors=True,
        handle_tool_error=True,
    )

# ── Knowledge Assessment Agent ───────────────────────────────────────────────
assessment_agent = make_agent(
    tools=[
        tool_retrieve, #(user queestion)
        tool_generate_quiz, #(generate quiz)
        tool_run_quiz, #adm
        tool_score, 
    ]
)

# ── Learning Path Agent ──────────────────────────────────────────────────────
learning_path_agent = make_agent(
    tools=[
        tool_analyse_gaps, 
    ]
)

# ── Adaptive Quiz Agent ──────────────────────────────────────────────────────
adaptive_quiz_agent = make_agent(
    tools=[
        tool_generate_quiz,
        tool_run_quiz,
        tool_score,
        tool_check_pass,
        tool_analyse_gaps,
    ]
)

# ── Mastery Tracking Agent ───────────────────────────────────────────────────
mastery_agent = make_agent(
    tools=[
        tool_update_mastery,
        tool_summary,
        tool_export_pdf,

    ]

)print("  mastery_agent         — 3 tools")

print("  adaptive_quiz_agent   — 5 tools")

print("All agents ready.")print("  learning_path_agent   — 1 tool")

print("  quick_study_agent     — 5 tools")print("  assessment_agent      — 4 tools")

All agents ready.
  quick_study_agent     — 5 tools
  assessment_agent      — 4 tools
  learning_path_agent   — 1 tool
  adaptive_quiz_agent   — 5 tools
  mastery_agent         — 3 tools


In [45]:
generate_quiz_questions("Financial Statements|advanced|3")

'[{"id": 1, "type": "conceptual", "difficulty": "easy", "question": "What is the primary objective of financial statements?", "options": {"A": "To assist the users in their decision-making", "B": "To provide information about economic resources only", "C": "To disclose accounting policies only", "D": "To provide information about cash flows only"}, "correct_answer": "A", "explanation": "The primary objective of financial statements is to assist the users in their decision-making by providing useful financial information."}, {"id": 2, "type": "scenario-based", "difficulty": "medium", "question": "A company\'s financial statement is used by trade associations to", "options": {"A": "Develop standard ratios and design uniform system of accounts", "B": "Evaluate the company\'s cash flows only", "C": "Determine the company\'s economic resources only", "D": "Predict the company\'s earning capacity only"}, "correct_answer": "A", "explanation": "Trade associations may analyse the financial stat

In [47]:
# SESSION

'de information about the earning capacity of the business:\nThey are to provide useful financial information which can gainfully\nbe utilised to predict, compare and evaluate the business firm’s earning\ncapacity.\n3. To provide information about cash flows:  They are to provide\ninformation useful to investors and creditors for predicting, comparing\nand evaluating, potential cash flows in terms of amount, timing and\nrelated uncertainties.\nReprint 2026-27\n147Financial Statements of a Company\n4. To ju\n\ntion to the shareholders\nin taking such important decisions.\n6. Aids trade associations in helping their members: Trade associations\nmay analyse the financial statements for the purpose of providing\nservice and protection to their members. They may develop standard\nratios and design uniform system of accounts.\n7. Helps stock exchanges: Financial statements help the stock exchanges\nto understand the extent of transparency in reporting on financial\nperformance and enables them to call for required\n\ntements of a Company 3\nLEARNING OBJECTIVES\nAfter studying this chapter,\nyou will be able to :\n• explain the nature and\nobjectives of financial\nstatements of a\ncompany;\n• describe the form and\ncontent of Statement of\nProfit and Loss of a\ncompany as per\nschedule III;\n• describe the form and\ncontent of balance sheet\nof a company as per\nschedule III;\n• explain the significance\nand limitations of\nfinancial statements;\nand\n• prepare the financial\nstatements.\nReprint 2026-27\n145Financial Statements of \n\ndge effectiveness of management: They supply information useful\nfor judging management’s ability to utilise the resources of a business\neffectively.\n5. Information about activities of business affecting the society:  They\nhave to report the activities of the business organisation affecting the\nsociety, which can be determined and described or measured and which\nare important in its social environment.\n6. Disclosing accounting policies:  These reports have to  provide the\nsignificant policies, co\n\ndecisions. Thus, the primary\nobjective of financial statements is to assist the users in their decision-making.\nThe specific objectives include the following:\n1. To provide information about economic resources and obligations of\na business:  They are prepared to provide adequate, reliable and\nperiodical information about economic resources and obligations of a\nbusiness firm to investors and other external parties who have limited\nauthority, ability or resources to obtain information.\n2. To provi\n\nhe concern, all\nthe obligations or liabilities payable to outsiders or creditors and claims of the\nowners on a particular date. It is one of the important statements depicting the\nfinancial position or status or strength of an undertaking.\nReprint 2026-27\n167Financial Statements of a Company\nStatement of Profit and Loss: The Statement of profit and loss is prepared for a\nspecific period to determine the operational results of an undertaking. It is a\nstatement of revenue earned and the expenses i\n\ne (net of expenses directly attributable to\nsuch income).\n3. Expense\nExpenses incurred to earn the income shown under various heads as discussed below:\n(a) Cost of Materials It applies to manufacturing companies. It consists of\nraw materials and other materials consumed in\nmanufacturing of goods.\n(b)Purchase of Stock-in-trade It means purchases of goods for the purpose of\ntrading.\nReprint 2026-27\n163Financial Statements of a Company\n(c) Changes in inventories of It is the difference between open\n\ns being\nfollowed. While deciding either cost of inventory or market value of\ninventory, many personal judgements are to be made based on certain\nconsiderations. Personal opinion, judgements and estimates are made\nwhile preparing the financial statements to avoid any possibility of\nover statement of assets and liabilities, income and expenditure,\nkeeping in mind the convention of conservatism.\nThus, financial statements are the summarised reports of recorded facts\nand are prepared the following a'


'[{"id": 1, "type": "conceptual", "difficulty": "easy", "question": "What is the primary purpose of a balance sheet?", "options": {"A": "To provide information about the earning capacity of a business", "B": "To provide information about the economic resources and obligations of a business", "C": "To provide information about cash flows", "D": "To provide information about the social activities of a business"}, "correct_answer": "B", "explanation": "The balance sheet provides a snapshot of a company\'s economic resources and obligations at a specific point in time."}, {"id": 2, "type": "scenario-based", "difficulty": "medium", "question": "A company has incurred expenses that have not yet been paid, but are expected to be paid in the future. How would these expenses be accounted for in the financial statements?", "options": {"A": "As cash outflows in the current period", "B": "As accruals in the current period", "C": "As prepayments in the current period", "D": "As non-current liabilities"}, "correct_answer": "B", "explanation": "Accruals are expenses that have been incurred but not yet paid, and are accounted for in the current period to match the expense with the related revenue."}, {"id": 3, "type": "definition", "difficulty": "easy", "question": "What is the term for the accounting method that recognizes revenues and expenses when they are earned or incurred, regardless of when cash is received or paid?", "options": {"A": "Cash basis accounting", "B": "Accrual basis accounting", "C": "Matching principle", "D": "GAAP"}, "correct_answer": "B", "explanation": "Accrual basis accounting recognizes revenues and expenses when they are earned or incurred, regardless of when cash is received or paid, which is a fundamental concept in financial accounting."}]'

In [48]:
get_summary = tool_summary.func("Financial Statements")
print(get_summary)

{"main_points":["Provide info on earning capacity", "Provide info on cash flows", "Help in decision-making for shareholders", "Aid trade associations and stock exchanges", "Disclose accounting policies", "Report activities affecting society", "Evaluate management effectiveness"], 
"summary":"Financial statements provide useful information about a company's economic resources, obligations, and earning capacity. They help investors, creditors, and shareholders make informed decisions and evaluate the company's financial performance. The primary objective of financial statements is to assist users in their decision-making by providing reliable and periodical information. Financial statements include the Statement of Profit and Loss and the Balance Sheet, which depict the financial position and operational results of a company.", 
"key_questions":["What are the primary objectives of financial statements?", "What information do financial statements provide to investors and creditors?", "How

In [ ]:
def make_agent(tools: list, system_note: str = "") -> AgentExecutor:
    """Helper: build a ReAct AgentExecutor with the given tools."""
    # Create agent directly without bind_tools - StructuredTool handles schema correctly
    agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)
    return AgentExecutor.from_agent_and_tools(
        agent=agent,
        tools=tools,
        verbose=False,
        max_iterations=20,
        handle_parsing_errors=True,
        handle_tool_error=True,
    )

def run_quick_study(name: str, topic: str):
    """Supervisor routes here for mode 1."""
    print("\n" + "="*60)
    print("  QUICK STUDY AGENT — running")
    print("="*60)

    prompt = f"""
You are a Quick Study Agent. Your job:
1. Create a learner profile for {name} studying {topic} at intermediate level
2. Retrieve study material for {topic} using retrieve_study_material
Complete all steps in order. Be thorough.
3. Generate 5 quiz questions using generate_quiz_questions
   with input "{topic}|intermediate|5"
4. Generate a study summary using generate_study_summary with input "{topic}"
 5. Export a quick study PDF using export_report_pdf with input "quick_study"
"""
    
    rest = f"""1. Create a learner profile for {name} studying {topic} at intermediate level
   using create_learner_profile with input "{name}|{topic}|intermediate"
2. Retrieve study material for {topic} using retrieve_study_material
3. Generate 5 quiz questions using generate_quiz_questions
   with input "{topic}|intermediate|5"
4. Generate a study summary using generate_study_summary with input "{topic}"
5. Export a quick study PDF using export_report_pdf with input "quick_study"

Complete all 5 steps in order. Be thorough.
2. Retrieve study material for {topic} using retrieve_study_material
# 3. Generate 5 quiz questions using generate_quiz_questions
#    with input "{topic}|intermediate|5"
# 4. Generate a study summary using generate_study_summary with input "{topic}"
# 5. Export a quick study PDF using export_report_pdf with input "quick_study"
# """
    result = quick_study_agent.invoke({"input": prompt})
    print("\nQuick Study complete.")
    print(f"Output: {result.get('output','')}")


# ── Quick Study Agent ────────────────────────────────────────────────────────
quick_study_agent = make_agent(
    tools=[
        # tool_search_profile,
        tool_create_profile,
        tool_retrieve, #(balance)
        tool_generate_quiz,
        tool_summary,
        tool_export_pdf,
    ]
)


In [33]:
!pip install --upgrade langchain-groq langchain-core pydantic

In [21]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="langchain_groq")


In [22]:
import os
os.environ["PYTHONWARNINGS"] = "ignore"

In [23]:
import warnings
import os

# 1. Block Python's standard warning display
warnings.filterwarnings("ignore")

# 2. Block Pydantic's specific deprecation messages
try:
    from pydantic import PydanticDeprecatedSince20
    warnings.filterwarnings("ignore", category=PydanticDeprecatedSince20)
except ImportError:
    pass

# 3. Tell the environment to stop Pydantic validator warnings
os.environ["PYDANTIC_SKIP_VALIDATOR_WARNINGS"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"


In [70]:
run_quick_study("Lila", "Balance SHeet components")


  QUICK STUDY AGENT — running
Retrieved 8 chunks for topic 'Balance Sheet components'.
Context preview:
017, prepare balance sheet in accordance to the Schedule III:
Reprint 2026-27
160 Accountancy : Company Accounts and Analysis of Financial Statements
Particulars Amount Particulars Amount
(Rs.) (Rs.)
Preliminary expenses 2,40,000 Goodwill 30,000
10% Debentures 2,00,000 Loose Tools 12,000
Stock in trade 1,40,000 Motor vehicles 4,75,000
Cash at bank 1,35,000 Provision for tax 16,000
Bills receivables 1,20,000
Solution:
Book of Shine and Bright Ltd.
Balance Sheet as at March 31, 2017
Particulars No...
{"main_points":["Equity and Liabilities include Shareholders’ Funds, Non-Current Liabilities, and Current Liabilities", "Shareholders’ Funds comprise Reserve and Surplus", "Non-Current Liabilities include Long Term Borrowings and Other Long Term Liabilities", "Current Liabilities include Short Term Loans and Advances", "Assets are classified into Non-Current Assets and Current Assets", "N

## Cell 6 — Supervisor + main entry point

The supervisor is not itself a ReAct agent — it's a simple router that reads the user's
mode choice and hands off to the right agent chain. This keeps it predictable.
Inside each agent, the LLM decides which tools to call and in what order.

In [ ]:
def run_quick_study(name: str, topic: str):
    """Supervisor routes here for mode 1."""
    print("\n" + "="*60)
    print("  QUICK STUDY AGENT — running")
    print("="*60)

    prompt = f"""
You are a Quick Study Agent. Your job:
1. FIRST: Search for an existing learner profile using search_learner_by_name with input "{name}"
   - If found (found=True), use that profile
   - If NOT found (found=False), create a new profile using create_learner_profile with input "{name}|{topic}|beginner"
2. Retrieve comprehensive study material for {topic} using retrieve_study_material (this gets all available chunks)
3. Generate 5 detailed quiz questions using generate_quiz_questions with input "{topic}|beginner|5"
4. Generate a detailed study summary using generate_study_summary with input "{topic}"
5. Export a comprehensive quick study PDF using export_report_pdf with input "quick_study"

Complete all 5 steps in order. Be thorough and use all available material.
"""
    result = quick_study_agent.invoke({"input": prompt})
    print("\nQuick Study complete.")
    print(f"Output: {result.get('output','')}")


def run_full_scale(name: str, topic: str, level: str):
    """Supervisor routes here for mode 2. Runs 4 agents in sequence."""

    # ── Stage 1: Create profile ──────────────────────────────────────────
    print("\n" + "="*60)
    print("  FULL SCALE — Stage 1: Knowledge Assessment")
    print("="*60)
    profile_json = create_learner_profile(f"{name}|{topic}|{level}")
    profile      = json.loads(profile_json)
    print(f"  Profile created: {profile}")

    # ── Stage 2: Knowledge Assessment Agent ─────────────────────────────
    assessment_prompt = f"""
You are a Knowledge Assessment Agent administering a diagnostic quiz.
The learner is {name}, studying {topic} at {level} level.

Your steps:
1. Retrieve study material: use retrieve_study_material with input "{topic}"
2. Generate 6 diagnostic questions: use generate_quiz_questions
   with input "{topic}|{level}|6"
3. Run the quiz interactively: use run_interactive_quiz with the questions JSON
4. Score and save the results: use score_and_save_quiz
   with input "diagnostic|1|<responses_json>"

Complete all steps. The goal is to identify the learner's baseline knowledge.
"""
    assessment_result = assessment_agent.invoke({"input": assessment_prompt})
    print(f"\n  Assessment complete.")

    # Extract responses from session for gap analysis
    # (score_and_save_quiz already stored them; fetch last session)
    conn  = sqlite3.connect(DB_PATH)
    row   = conn.execute(
        "SELECT responses_json, score_pct, passed FROM quiz_sessions "
        "WHERE learner_id=? AND session_type='diagnostic' ORDER BY created_at DESC LIMIT 1",
        (profile["id"],)
    ).fetchone()
    conn.close()

    if not row:
        print("  Warning: could not retrieve diagnostic responses. Using empty gaps.")
        responses_json = "[]"
        score_pct, passed = 0.0, False
    else:
        responses_json, score_pct, passed = row[0], row[1], bool(row[2])

    print(f"  Diagnostic score: {score_pct*100:.0f}% — {'passed' if passed else 'not passed'}")

    # ── Stage 3: Learning Path Agent ────────────────────────────────────
    print("\n" + "-"*60)
    print("  FULL SCALE — Stage 3: Learning Path")
    print("-"*60)

    gap_prompt = f"""
You are a Learning Path Agent.
Analyse these quiz responses and identify the learner's knowledge gaps.
Use analyse_knowledge_gaps with this responses JSON:
{responses_json}

Produce an ordered learning path showing what the learner needs to study.
"""
    learning_path_agent.invoke({"input": gap_prompt})
    print("  Learning path generated.")

    # ── Stage 4: Adaptive Quiz Agent ────────────────────────────────────
    print("\n" + "-"*60)
    print("  FULL SCALE — Stage 4: Adaptive Quiz")
    print("-"*60)
    print(f"  Goal: reach {int(PASS_THRESHOLD*100)}% or above.")
    print("  The agent will loop, changing questions each round, until you pass.\n")

    gap_data     = SESSION.get("gap_data", {})
    missed       = gap_data.get("gaps", [])
    missed_str   = ", ".join(missed[:4]) if missed else ""

    adaptive_prompt = f"""
You are an Adaptive Quiz Agent. Your goal is to help {name} pass the quiz on {topic}
by scoring {int(PASS_THRESHOLD*100)}% or higher.

Rules:
- Generate 5 questions, run them, score them, then check_pass_status
- If NOT PASSED: generate NEW questions focused on the missed concepts and repeat
- If PASSED: stop the loop and report success
- Maximum {MAX_QUIZ_ROUNDS} rounds
- Each round MUST use generate_quiz_questions, then run_interactive_quiz,
  then score_and_save_quiz, then check_pass_status

Start with round 1. Focus areas from diagnostic: {missed_str if missed_str else 'general ' + topic}

Use generate_quiz_questions with input "{topic}|{level}|5|{missed_str}"
"""
    adaptive_result = adaptive_quiz_agent.invoke({"input": adaptive_prompt})
    print("\n  Adaptive quiz complete.")

    # ── Stage 5: Mastery Tracking Agent ─────────────────────────────────
    print("\n" + "-"*60)
    print("  FULL SCALE — Stage 5: Mastery Tracking & Report")
    print("-"*60)

    # Get best score from DB
    conn = sqlite3.connect(DB_PATH)
    best_row = conn.execute(
        "SELECT MAX(score_pct), MAX(passed) FROM quiz_sessions "
        "WHERE learner_id=? AND session_type='adaptive'",
        (profile["id"],)
    ).fetchone()
    conn.close()
    best_score = best_row[0] if best_row and best_row[0] else score_pct
    best_passed = bool(best_row[1]) if best_row and best_row[1] is not None else False

    mastery_prompt = f"""
You are a Mastery Tracking Agent. The learner {name} has completed the adaptive quiz on {topic}.
Best score: {best_score*100:.0f}%. Passed: {best_passed}.

Your steps:
1. Update the mastery record: use update_mastery_record
   with input "{best_score}|{best_passed}"
2. Generate a study summary: use generate_study_summary with input "{topic}"
3. Export the mastery report PDF: use export_report_pdf with input "mastery_report"

Complete all 3 steps.
"""
    mastery_agent.invoke({"input": mastery_prompt})
    print("\n  Mastery report complete.")




In [50]:

# ═══════════════════════════════════════════════════════════════════════════
# ── SUPERVISOR / ENTRY POINT ───────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════

def run_tutor():
    print("\n" + "="*60)
    print("  ADAPTIVE BUSINESS MANAGEMENT TUTOR")
    print("  Multi-Agent System (LangChain ReAct)")
    print("="*60)

    name  = "Nandita"#input("\nYour name: ").strip() or "Learner"
    topic = "Finance"#input("Topic to study (e.g. Financial Statements, Marketing Mix): ").strip()
    if not topic:
        topic = "Business Management"

    print("\nStudy mode:")
    print("  1  Quick Study  — summary + quiz exported to PDF  (~2 min)")
    print("  2  Full Scale   — diagnostic → gaps → adaptive quiz → mastery report")
    mode = "1"#input("\nChoose (1 or 2): ").strip()

    if mode == "1":
        run_quick_study(name, topic)

    # elif mode == "2":
    #     print("\nExperience level:")
    #     print("  1  beginner  |  2  intermediate  |  3  advanced")
    #     lv = input("Choose (1/2/3) [1]: ").strip()
    #     level = {"1":"beginner","2":"intermediate","3":"advanced"}.get(lv, "beginner")
    #     run_full_scale(name, topic, level)

    # else:
    #     print("Invalid choice. Please enter 1 or 2.")

    print("\n" + "="*60)
    print("  Session complete. Check the outputs/ folder for your PDF.")
    print("="*60)



In [51]:

# ── START ──
run_tutor()


  ADAPTIVE BUSINESS MANAGEMENT TUTOR
  Multi-Agent System (LangChain ReAct)

Study mode:
  1  Quick Study  — summary + quiz exported to PDF  (~2 min)
  2  Full Scale   — diagnostic → gaps → adaptive quiz → mastery report

  QUICK STUDY AGENT — running


c:\Users\Kurinchi1\anaconda3\Lib\site-packages\executing\executing.py:713: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  right=ast.Str(s=sentinel),
c:\Users\Kurinchi1\anaconda3\Lib\ast.py:587: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  return Constant(*args, **kwargs)
c:\Users\Kurinchi1\anaconda3\Lib\site-packages\executing\executing.py:713: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  right=ast.Str(s=sentinel),
c:\Users\Kurinchi1\anaconda3\Lib\ast.py:587: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  return Constant(*args, **kwargs)


BadRequestError: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

## Cell 7 — Inspect stored records (optional)

In [ ]:
import pandas as pd

conn = sqlite3.connect(DB_PATH)

print("── Learner profiles ──")
df1 = pd.read_sql("SELECT * FROM learner_profiles ORDER BY created_at DESC LIMIT 10", conn)
print(df1.to_string(index=False) if not df1.empty else "  (none yet)")

print("\n── Quiz sessions ──")
df2 = pd.read_sql(
    "SELECT learner_id,session_type,topic,round_number,score_pct,passed,created_at "
    "FROM quiz_sessions ORDER BY created_at DESC LIMIT 20", conn
)
if not df2.empty:
    df2["score_pct"] = (df2["score_pct"]*100).round(0).astype(int).astype(str) + "%"
    print(df2.to_string(index=False))
else:
    print("  (none yet)")

print("\n── Mastery records ──")
df3 = pd.read_sql("SELECT * FROM mastery_records ORDER BY updated_at DESC", conn)
if not df3.empty:
    df3["best_score"] = (df3["best_score"]*100).round(0).astype(int).astype(str) + "%"
    print(df3.to_string(index=False))
else:
    print("  (none yet)")

conn.close()
